# Tarea 8. CNN para clasificar rostros de estudiantes

**Materia:** Inteligencia Artificial

**Equipo:** Miriam Alejandra Bazaldua Machado, Daniel Jimenez Duarte, José de Jesús Quintero Noris, Rodrigo Villegas Valdez

En este cuaderno entrenamos una **red neuronal convolucional (CNN)** desde cero con el dataset de rostros que
armamos en la Tarea 6. La red aprende a decir de quién es cada cara. Al final se guarda el modelo en un archivo
para usarlo en un programa local (`reconocer_rostros.py`) que identifica a las personas con la cámara en tiempo real.

## Cómo ejecutarlo en Google Colab

1. Entra a https://colab.research.google.com y sube este cuaderno (**Archivo > Subir cuaderno**).
2. Ten a la mano el archivo `fotos.zip` con el dataset. Adentro debe haber **una carpeta por persona** con sus imágenes.
3. Ejecuta las celdas en orden, con el botón de **Ejecutar** (el triángulo) de cada una.
4. Para que entrene más rápido, activa la GPU en **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU**.
   También funciona sin GPU, solo tarda más.

## 1. Librerías

In [ ]:
import os
import json
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Semilla fija para que los resultados se parezcan entre ejecuciones
keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

## 2. Subir y descomprimir el dataset

Al ejecutar la celda aparece un botón para elegir el archivo `fotos.zip`. Colab lo descomprime en la carpeta
`dataset_extraido`. Las carpetas de Colab se borran cuando se cierra la sesión, así que si la reinicias hay que
volver a subir el zip.

In [ ]:
from google.colab import files

print("Selecciona el archivo 'fotos.zip':")
subido = files.upload()
nombre_zip = list(subido.keys())[0]

with zipfile.ZipFile(nombre_zip, 'r') as zip_ref:
    zip_ref.extractall('dataset_extraido')
print('Dataset descomprimido')

In [ ]:
EXTENSIONES = ('.jpg', '.jpeg', '.png', '.bmp')


def buscar_raiz(base):
    # Busca la carpeta que tiene adentro una subcarpeta por persona
    for actual, carpetas, _ in os.walk(base):
        carpetas[:] = [c for c in carpetas if not c.startswith(('__', '.'))]
        con_fotos = [c for c in carpetas
                     if any(f.lower().endswith(EXTENSIONES) for f in os.listdir(os.path.join(actual, c)))]
        if len(con_fotos) >= 2:
            return actual
    raise RuntimeError('No encontré carpetas con imágenes. Revisa la estructura del zip.')


RUTA_DATASET = buscar_raiz('dataset_extraido')
print('Dataset en:', RUTA_DATASET)

# Cuenta las imágenes de cada persona
personas = sorted(c for c in os.listdir(RUTA_DATASET) if os.path.isdir(os.path.join(RUTA_DATASET, c)) and not c.startswith(('__', '.')))
for persona in personas:
    cantidad = len([f for f in os.listdir(os.path.join(RUTA_DATASET, persona)) if f.lower().endswith(EXTENSIONES)])
    print(f'{persona:25s} {cantidad} imágenes')

## 3. Preparar las imágenes (con distinta iluminación)

El problema más común en reconocimiento de rostros es la **luz**: la misma cara con más o menos brillo puede dar
otra predicción. Para atacarlo hacemos dos cosas:

1. **Variantes de luz:** de cada foto generamos 5 copias más oscuras o más claras (corrección *gamma* de 0.5 a 2.0).
   Así la red ve a cada persona con distintas iluminaciones.
2. **Normalización de iluminación (CLAHE):** a todas las imágenes se les iguala el contraste local, sobre el canal de
   luminosidad. Esto **se aplica igual en este cuaderno y en el programa de la cámara**, para que la red siempre
   vea imágenes parecidas sin importar la luz del cuarto.

Además, el 80 % / 20 % se separa **por foto original**, no por copia: las 5 variantes de una misma foto van todas a
entrenamiento o todas a validación. Si no, la validación saldría artificialmente alta porque vería la misma foto con
otro brillo.

Todas las imágenes se ajustan a **128 × 128 píxeles** y a color (RGB).

In [ ]:
import cv2
import random
import shutil

TAMANO = 128
LOTE = 32
GAMMAS = [0.5, 0.75, 1.0, 1.5, 2.0]   # <1 aclara, >1 oscurece


def ajustar_gamma(imagen, gamma):
    # Cambia el brillo con una tabla de 256 valores
    tabla = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype('uint8')
    return cv2.LUT(imagen, tabla)


def normalizar_iluminacion(imagen_bgr):
    # CLAHE sobre el canal de luminosidad (L) para igualar el contraste local
    lab = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)


def leer(ruta):
    return cv2.imdecode(np.fromfile(ruta, dtype=np.uint8), cv2.IMREAD_COLOR)


def guardar(ruta, imagen):
    cv2.imencode('.jpg', imagen)[1].tofile(ruta)


DESTINO = 'dataset_preparado'
if os.path.exists(DESTINO):
    shutil.rmtree(DESTINO)

random.seed(42)
for persona in personas:
    fotos = sorted(f for f in os.listdir(os.path.join(RUTA_DATASET, persona)) if f.lower().endswith(EXTENSIONES))
    random.shuffle(fotos)
    n_val = max(1, int(len(fotos) * 0.2))
    for numero, nombre in enumerate(fotos):
        imagen = leer(os.path.join(RUTA_DATASET, persona, nombre))
        if imagen is None:
            continue
        imagen = cv2.resize(imagen, (TAMANO, TAMANO))
        grupo = 'val' if numero < n_val else 'train'
        carpeta = os.path.join(DESTINO, grupo, persona)
        os.makedirs(carpeta, exist_ok=True)
        for gamma in GAMMAS:
            variante = normalizar_iluminacion(ajustar_gamma(imagen, gamma))
            guardar(os.path.join(carpeta, f'{numero:04d}_g{gamma}.jpg'), variante)

print('Imágenes preparadas en', DESTINO)

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    os.path.join(DESTINO, 'train'), seed=42, label_mode='categorical',
    image_size=(TAMANO, TAMANO), batch_size=LOTE)

val_ds = keras.utils.image_dataset_from_directory(
    os.path.join(DESTINO, 'val'), shuffle=False, label_mode='categorical',
    image_size=(TAMANO, TAMANO), batch_size=LOTE)

clases = train_ds.class_names
rutas_val = list(val_ds.file_paths)   # se guardan para medir la precisión por nivel de luz
print('Clases:', clases)

# Muestra algunas imágenes de entrenamiento con su etiqueta
figura = plt.figure(figsize=(10, 6))
for imagenes, etiquetas in train_ds.take(1):
    for i in range(min(12, len(imagenes))):
        eje = plt.subplot(3, 4, i + 1)
        eje.imshow(imagenes[i].numpy().astype('uint8'))
        eje.set_title(clases[int(np.argmax(etiquetas[i]))], fontsize=9)
        eje.axis('off')
plt.tight_layout()
plt.show()

# Prepara los datos para que el entrenamiento sea más rápido
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 4. Modelo (CNN)

Se puede elegir entre dos formas de construir la CNN con la variable `MODO`:

- **`'transfer'` (Transfer Learning, recomendado):** se usa **MobileNetV2**, una CNN que ya fue entrenada con millones
  de imágenes (ImageNet) y que ya sabe detectar bordes, texturas y formas. Se le quita la capa final y se le pone
  una nueva con una salida por persona. Funciona mucho mejor cuando hay pocas fotos por persona.
- **`'scratch'` (desde cero):** una CNN diseñada por nosotros, con 4 bloques de convolución.

En ambas se usa:

- **Aumento de datos** (giro, rotación, zoom y un poco de contraste): modifica un poco las imágenes al azar mientras
  se entrena. Solo actúa en el entrenamiento. El brillo no se altera aquí porque las variantes de luz ya se generaron
  en la sección 3.
- **Normalización de píxeles:** `Rescaling` dentro del modelo. En la CNN desde cero pasa de 0-255 a [0, 1]; en
  MobileNetV2 a [-1, 1], que es el rango con el que se entrenó.
- **Conv2D + BatchNormalization + MaxPooling2D** (en la CNN desde cero): cada `Conv2D` detecta rasgos, la
  `BatchNormalization` estabiliza los valores para que entrene mejor y cada `MaxPooling2D` reduce la imagen a la mitad.
- **Dropout:** apaga neuronas al azar en cada paso para evitar el sobreajuste.
- **Dense (softmax):** una salida por persona, con la probabilidad de que la cara sea de ella.

In [ ]:
MODO = 'transfer'    # 'transfer' = MobileNetV2 preentrenada | 'scratch' = CNN desde cero

aumento = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='aumento_de_datos')

if MODO == 'transfer':
    base = keras.applications.MobileNetV2(
        input_shape=(TAMANO, TAMANO, 3), include_top=False, weights='imagenet')
    base.trainable = False                       # primero solo se entrena la capa nueva

    entrada = keras.Input(shape=(TAMANO, TAMANO, 3))
    x = aumento(entrada)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # pixeles de 0-255 a [-1, 1]
    x = base(x, training=False)                  # BatchNorm de la base en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)                   # evita que se memorice los datos
    salida = layers.Dense(len(clases), activation='softmax')(x)   # una salida por persona
    modelo = keras.Model(entrada, salida)
else:
    modelo = keras.Sequential([
        keras.Input(shape=(TAMANO, TAMANO, 3)),
        aumento,
        layers.Rescaling(1.0 / 255),             # pixeles de 0-255 a [0, 1]

        layers.Conv2D(32, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dropout(0.5),                     # evita que se memorice los datos
        layers.Dense(128, activation='relu'),
        layers.Dense(len(clases), activation='softmax'),
    ])

modelo.summary()

## 5. Entrenamiento

- **Optimizador Adam** y pérdida **`categorical_crossentropy`** (las etiquetas están en one-hot, un valor por persona).
- **EarlyStopping:** si la precisión de validación no mejora en varias épocas seguidas, el entrenamiento se detiene solo
  y se queda con los mejores pesos.
- **ModelCheckpoint:** cada vez que la precisión de validación mejora, guarda el modelo en `mejor_modelo.keras`.
- En modo `'transfer'` hay **dos fases**: primero se entrena solo la capa nueva (la base queda congelada) y después se
  hace *fine-tuning*: se descongelan las últimas capas de MobileNetV2 y se siguen entrenando con una tasa de
  aprendizaje muy pequeña para ajustarlas a nuestras caras.

In [ ]:
def callbacks(paciencia):
    return [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=paciencia, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint('mejor_modelo.keras', monitor='val_accuracy', save_best_only=True),
    ]


if MODO == 'transfer':
    # Fase 1: solo la capa nueva
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss='categorical_crossentropy', metrics=['accuracy'])
    fase1 = modelo.fit(train_ds, validation_data=val_ds, epochs=15,
                       callbacks=callbacks(5), verbose=2)

    # Fase 2: fine-tuning de las ultimas capas de MobileNetV2
    base.trainable = True
    for capa in base.layers[:-30]:
        capa.trainable = False
    modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
                   loss='categorical_crossentropy', metrics=['accuracy'])
    fase2 = modelo.fit(train_ds, validation_data=val_ds, epochs=15,
                       callbacks=callbacks(5), verbose=2)
    historia = {k: fase1.history[k] + fase2.history[k] for k in fase1.history}
else:
    modelo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    entrenamiento = modelo.fit(train_ds, validation_data=val_ds, epochs=40,
                               callbacks=callbacks(8), verbose=2)
    historia = entrenamiento.history

# Carga el mejor modelo guardado por ModelCheckpoint
modelo = keras.models.load_model('mejor_modelo.keras')

## 6. Resultados

Si la precisión de entrenamiento sube pero la de validación se queda atrás, la red se está sobreajustando.

In [ ]:
figura, (eje1, eje2) = plt.subplots(1, 2, figsize=(11, 4))

eje1.plot(historia['accuracy'], label='Entrenamiento')
eje1.plot(historia['val_accuracy'], label='Validación')
eje1.set_title('Precisión por época')
eje1.set_xlabel('Época')
eje1.set_ylabel('Precisión')
eje1.legend()
eje1.grid(True, linestyle='--', alpha=0.5)

eje2.plot(historia['loss'], label='Entrenamiento')
eje2.plot(historia['val_loss'], label='Validación')
eje2.set_title('Pérdida por época')
eje2.set_xlabel('Época')
eje2.set_ylabel('Pérdida')
eje2.legend()
eje2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Predice todo el conjunto de validación (verdad y predicción se sacan en la misma pasada)
verdad, prediccion = [], []
for imagenes, etiquetas in val_ds:
    probabilidades = modelo(imagenes, training=False).numpy()
    verdad.extend(np.argmax(etiquetas.numpy(), axis=1))
    prediccion.extend(np.argmax(probabilidades, axis=1))

verdad = np.array(verdad)
prediccion = np.array(prediccion)
precision = np.mean(verdad == prediccion)
print(f'Precisión en validación: {precision:.2%} ({len(verdad)} imágenes)')
print()
print(classification_report(verdad, prediccion, labels=range(len(clases)), target_names=clases, zero_division=0))

### Precisión según la iluminación

Si la red es robusta a la luz, la precisión debe ser parecida con imágenes oscuras (gamma 2.0) y claras (gamma 0.5).
Si hay una diferencia grande, hacen falta más fotos con esa luz.

In [ ]:
gamma_de_cada_imagen = np.array([float(r.split('_g')[-1].replace('.jpg', '')) for r in rutas_val])
for gamma in GAMMAS:
    mascara = gamma_de_cada_imagen == gamma
    acierto = np.mean(verdad[mascara] == prediccion[mascara])
    print(f'gamma {gamma:<4} ({"más claro" if gamma < 1 else "más oscuro" if gamma > 1 else "original "}): {acierto:.2%}')

In [ ]:
# Matriz de confusión: filas = persona real, columnas = lo que predijo la red
matriz = confusion_matrix(verdad, prediccion, labels=range(len(clases)))
figura, eje = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matriz, display_labels=clases).plot(ax=eje, cmap='Blues', colorbar=False, xticks_rotation=45)
eje.set_title('Matriz de confusión (validación)')
plt.tight_layout()
plt.show()

## 7. Guardar el modelo

Se guardan dos archivos que necesita el programa local:

- `modelo_cnn_rostros.keras`: la red ya entrenada (arquitectura y pesos).
- `modelo_info.json`: los nombres de las personas en el orden que usa la red, el tamaño de imagen y el dato de que se usa CLAHE (el programa lo lee para aplicar la misma normalización de luz).

En Colab se descargan a la computadora. Hay que copiarlos a la misma carpeta que `reconocer_rostros.py`.

In [ ]:
modelo.save('modelo_cnn_rostros.keras')   # mejor modelo (ya cargado desde el checkpoint)

with open('modelo_info.json', 'w', encoding='utf-8') as archivo:
    json.dump({'clases': clases, 'tamano': TAMANO, 'modo': MODO,
               'normalizar_iluminacion': {'metodo': 'clahe_lab', 'clip': 2.0, 'tile': 8}},
              archivo, ensure_ascii=False, indent=2)

print('Archivos guardados: modelo_cnn_rostros.keras y modelo_info.json')

# En Colab los descarga a la computadora
try:
    from google.colab import files
    files.download('modelo_cnn_rostros.keras')
    files.download('modelo_info.json')
except ImportError:
    print('No estoy en Colab, los archivos quedaron en la carpeta actual')

## Conclusiones

In [ ]:
# Resumen con los números de esta ejecución
por_persona = matriz.diagonal() / np.maximum(matriz.sum(axis=1), 1)
mejor = int(np.argmax(por_persona))
peor = int(np.argmin(por_persona))
epocas = len(historia['loss'])
mejor_train = max(historia['accuracy'])

print(f'La red se entrenó {epocas} épocas y tiene {modelo.count_params():,} parámetros.')
print(f'Precisión en validación: {precision:.2%}. En entrenamiento llegó a {mejor_train:.2%}.')
if por_persona[mejor] == por_persona[peor]:
    print(f'Reconoce a todas las personas con la misma precisión ({por_persona[mejor]:.2%}).')
else:
    print(f'La persona que mejor reconoce es {clases[mejor]} ({por_persona[mejor]:.2%}) y la que más le cuesta es {clases[peor]} ({por_persona[peor]:.2%}).')

**Importante para interpretar el resultado:** las fotos de cada integrante se tomaron en ráfaga con la cámara, así que
muchas se parecen entre sí. Por eso, al separar al azar el 20 % para validar, es probable que haya fotos casi
idénticas en entrenamiento y validación, y la precisión salga más alta que en la vida real. La prueba verdadera es el
programa con la cámara en tiempo real (`reconocer_rostros.py`), donde la red ve caras nuevas, con otra luz y otra postura.

Si en la cámara se confunde, lo que más ayuda es tomar más fotos con distintas condiciones de luz (ventana, lámpara,
sombra), ángulos y expresiones, y volver a entrenar. Las variantes de gamma simulan la luz, pero no reemplazan fotos reales.